**Machine Learning Lab**

# 12. Transfer Learning and Hyperparameter Tuning

**Transfer Learning** and **Hyperparameter Tuning** are two essential concepts in modern machine learning. Transfer Learning involves leveraging a pre-trained model on one task and adapting it for a different but related task, significantly reducing the time and data needed to train a model. Hyperparameter Tuning, on the other hand, focuses on optimizing the settings that govern how a model learns, such as learning rate, batch size, and the number of epochs, to improve performance. Together, these techniques help build more efficient and effective machine learning models.

This notebook reuses the chess pieces dataset from the CNN notebook, but instead of training a CNN end-to-end, we use a pretrained **ResNet50** purely as a fixed **feature extractor** (no fine-tuning), then train a conventional classifier (Random Forest) on those features — and tune that classifier's hyperparameters automatically with **Optuna**.

## Initialization

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

## Downloading the Dataset

This notebook uses the same [chess pieces dataset](https://www.kaggle.com/datasets/anshulmehtakaggl/chess-pieces-detection-images-dataset) as the CNN notebook, downloaded fresh here via the Kaggle API (rather than reading it back out of Google Drive, as in an earlier version of this notebook — that meant this notebook could only run for whoever originally saved the dataset to their own Drive). You'll need a Kaggle API token:

1. Log into Kaggle, go to your [account settings](https://www.kaggle.com/settings).
2. Under "API", click "Create New Token" — this downloads a file called `kaggle.json`.
3. Run the cell below and upload that `kaggle.json` file when prompted.

In [ ]:
!pip install -q kaggle

from google.colab import files
print("Upload your kaggle.json (Kaggle account -> Settings -> API -> Create New Token)")
uploaded = files.upload()  # select kaggle.json

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

Download and unzip the dataset.

In [ ]:
!kaggle datasets download -d anshulmehtakaggl/chess-pieces-detection-images-dataset -p . --unzip -q
!mv 'Chess Pieces Detection Images Dataset' Chess 2>/dev/null || true

Set the image size and batch size to use throughout.

In [ ]:
# Path to chess piece images
dataset_path = 'Chess'
img_size = (224, 224)
batch_size = 32
seed = 42

Load every image, inferring class labels from the subfolder names.

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Load full dataset
full_ds = tf.keras.preprocessing.image_dataset_from_directory(
    dataset_path,
    image_size=img_size,
    batch_size=batch_size,
    seed=seed,
    shuffle=True  # Shuffle before splitting
)

## Train-test Split

Hold out 20% of the batches as a test set that we won't touch again until final evaluation.

In [ ]:
# Determine the size of the full dataset
dataset_size = tf.data.experimental.cardinality(full_ds).numpy()

# Calculate the number of batches for training and validation
train_size = int(0.8 * dataset_size)

# Split the dataset
train_ds = full_ds.take(train_size)
test_ds = full_ds.skip(train_size)

## Removing Imbalance

Like most real-world image datasets, this one probably doesn't have exactly the same number of images per chess piece type. Count how many training examples belong to each class.

In [ ]:
import tensorflow as tf
from collections import defaultdict

# Step 1: Count examples per class
class_counts = defaultdict(int)

for _, label_batch in train_ds: # Rename 'label' to 'label_batch' for clarity
    # Iterate through each individual label in the batch
    for label in label_batch.numpy():
        class_counts[int(label)] += 1

print("Class counts:", class_counts)

We'll oversample every other class up to the size of the largest one.

In [ ]:
# Step 2: Find the maximum class size
max_count = max(class_counts.values())

To oversample, we first need every image and label pulled out of the batched dataset and grouped by class.

In [ ]:
# Step 3: Split dataset by class into separate lists of tensors
class_images = defaultdict(list)
class_labels = defaultdict(list)

# Iterate through the dataset in batches
for image_batch, label_batch in train_ds:
    # Iterate through each image and label within the current batch
    for image, label in zip(tf.unstack(image_batch), tf.unstack(label_batch)):
        # Explicitly cast image to float32 and label to int64
        class_images[int(label.numpy())].append(tf.cast(image, tf.float32))
        class_labels[int(label.numpy())].append(tf.cast(label, tf.int64))

For each class, build a small `tf.data.Dataset` and repeat it enough times (plus a partial extra pass) so every class ends up with `max_count` examples — simple oversampling by duplication.

In [ ]:
# Step 4: Convert to tf.data.Dataset and oversample
balanced_datasets = []
for label in class_counts.keys():
    # Get the lists of images and labels for this class
    items_images = class_images[label]
    items_labels = class_labels[label]

    # Ensure both lists are converted to tensors or numpy arrays for from_tensor_slices
    # Converting to NumPy arrays is often safer with from_tensor_slices for lists of tensors
    items_images_np = tf.stack(items_images).numpy()
    items_labels_np = tf.stack(items_labels).numpy()

    # Create the dataset from separate arrays
    ds = tf.data.Dataset.from_tensor_slices((items_images_np, items_labels_np))

    # Continue with oversampling as before
    repeats = max_count // len(items_images)
    remainder = max_count % len(items_images)

    # Repeat whole dataset and add a few more
    oversampled_ds = ds.repeat(repeats)
    oversampled_ds = oversampled_ds.concatenate(ds.take(remainder))
    balanced_datasets.append(oversampled_ds)

Combine every class's oversampled dataset back into one, then shuffle and batch it.

In [ ]:
# Step 5: Combine and shuffle
balanced_train_ds = balanced_datasets[0]
for ds in balanced_datasets[1:]:
    balanced_train_ds = balanced_train_ds.concatenate(ds)

shuffle_buffer_size = tf.data.experimental.cardinality(balanced_train_ds).numpy()

balanced_train_ds = balanced_train_ds.shuffle(shuffle_buffer_size).batch(32).prefetch(tf.data.AUTOTUNE)

## Data Augmentation

On top of oversampling, we also apply the same kind of random augmentation used in the CNN notebook, so the duplicated images from oversampling aren't literally identical copies every time the model sees them.

In [ ]:
from tensorflow.keras.layers import RandomFlip, RandomRotation, RandomZoom, RandomBrightness, RandomContrast

data_augmentation = tf.keras.Sequential([
    RandomFlip("horizontal"),
    RandomRotation(0.2),
    RandomZoom(0.2),
    RandomBrightness(0.2),
    RandomContrast(0.2),
])

Apply the augmentation to the balanced training set.

In [ ]:
# Apply data augmentation to the balanced training dataset
augmented_train_ds = balanced_train_ds.map(lambda x, y: (data_augmentation(x, training=True), y))
augmented_train_ds = augmented_train_ds.prefetch(tf.data.AUTOTUNE)

Double check the class counts are now balanced after oversampling.

In [ ]:
# Show number of samples in augmented training set
augmented_class_counts = defaultdict(int)

# Iterate through the augmented_train_ds in batches
for _, label_batch in augmented_train_ds:
    for label in label_batch.numpy():
        augmented_class_counts[int(label)] += 1

print("Class counts in augmented_train_ds:", augmented_class_counts)

Visualize a sample grid to sanity-check the augmented, balanced images.

In [ ]:
import matplotlib.pyplot as plt

image_batch, label_batch = next(iter(augmented_train_ds))

num_images_to_display = 25

fig, axes = plt.subplots(5, 5, figsize=(10, 10))

# Iterate over the images in the batch
for i in range(num_images_to_display):
  ax = axes[i // 5, i % 5]
  ax.imshow(image_batch[i].numpy().astype("uint8"))
  ax.set_title(f"Class: {label_batch[i].numpy()}")
  ax.axis("off")

plt.tight_layout()
plt.show()

## Feature Extraction using Transfer Learning Model

Rather than training a CNN end-to-end, we load **ResNet50** (again pretrained on ImageNet) without its classification head, and use it purely to turn each image into a compact feature vector via `GlobalAveragePooling2D`. We never train ResNet50 itself here — it's used exactly as-is.

In [ ]:
import numpy as np
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras import Model

# Load the ResNet50 model without the top (classification) layers
resnet50 = ResNet50(weights='imagenet', include_top=False, input_shape=(img_size[0], img_size[1], 3))

# Add a Global Average Pooling layer
x = GlobalAveragePooling2D()(resnet50.output)

# Create the feature extraction model using ResNet50
feature_extractor_resnet = Model(inputs=resnet50.input, outputs=x)

# Function to extract features
def extract_features(dataset, feature_model):
  features = []
  labels = []
  # Iterate through the dataset in batches
  for images, batch_labels in dataset:
    batch_features = feature_model.predict(images, verbose=0)
    features.extend(batch_features)
    labels.extend(batch_labels.numpy())
  return np.array(features), np.array(labels)

Run every training and test image through ResNet50 to get their feature vectors.

In [ ]:
# Extract features from augmented training set using ResNet50
print("Extracting features from the augmented training set using ResNet50...")
X_train, y_train = extract_features(augmented_train_ds, feature_extractor_resnet)
print("Features extracted from augmented training set using ResNet50.")

# Extract features from test set using ResNet50
print("Extracting features from the test set using ResNet50...")
X_test, y_test = extract_features(test_ds, feature_extractor_resnet)
print("Features extracted from test set using ResNet50.")

X_train

Check the shape of the extracted feature matrix — one row per image, 2048 ResNet50 features per row.

In [ ]:
X_train.shape

## Baseline Classifier

Before tuning anything, train a Random Forest with default settings directly on the ResNet50 features, as a baseline to compare tuning against later.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

rf_classifier = RandomForestClassifier(random_state=42)
rf_classifier.fit(X_train, y_train)

Evaluate the baseline on the test set.

In [ ]:
test_predictions = rf_classifier.predict(X_test)

accuracy = accuracy_score(y_test, test_predictions)
print(f"Accuracy: {accuracy:.4f}")

print("\nClassification Report:")
class_names = full_ds.class_names
print(classification_report(y_test, test_predictions, target_names=class_names))

## Hyperparameter Tuning using Optuna

[Optuna](https://optuna.org/) automatically searches a hyperparameter space, using the results of earlier trials to make smarter guesses about which combinations to try next (rather than a blind grid or random search).

**Important:** the objective function needs its own train/validation split, carved out of `X_train`/`y_train` — it must **not** evaluate on `X_test` for every one of its trials. If it did, `X_test` would effectively be used to *choose* the hyperparameters, so the final reported test accuracy would be optimistically biased (the model would have been indirectly tuned to do well on exactly that test set). We hold `X_test` out completely until the very last cell, where the final tuned model is evaluated on it exactly once.

In [ ]:
try:
  import optuna
except:
  !pip install optuna
  import optuna

from sklearn.model_selection import train_test_split

# Carve a validation set out of the training data, just for hyperparameter search.
# X_test / y_test stay completely unused until the final evaluation cell below.
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)

def objective(trial):
    # Define hyperparameter search space
    n_estimators = trial.suggest_int('n_estimators', 50, 500)
    max_depth = trial.suggest_int('max_depth', 5, 50)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 20)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 20)
    criterion = trial.suggest_categorical('criterion', ['gini', 'entropy'])

    # Create and train the Random Forest Classifier with suggested hyperparameters
    rf_classifier = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        criterion=criterion,
        random_state=42,
        n_jobs=-1  # Use all available cores
    )
    rf_classifier.fit(X_tr, y_tr)

    # Evaluate the model on the held-out validation set (not the test set!)
    val_predictions = rf_classifier.predict(X_val)
    accuracy = accuracy_score(y_val, val_predictions)

    # Return the metric to minimize or maximize (accuracy in this case)
    return accuracy

# Create an Optuna study
study = optuna.create_study(direction='maximize') # We want to maximize accuracy

# Run the optimization. This can take a while - lower n_trials if you want faster (rougher)
# results while experimenting.
study.optimize(objective, n_trials=150) # Run 150 trials

# Print the best hyperparameters and the corresponding best accuracy (on the validation set)
print("Best hyperparameters found by Optuna:")
print(study.best_params)
print("Best validation accuracy:", study.best_value)

Finally, retrain a model with the best hyperparameters found — this time on the **full** `X_train`/`y_train` (validation data included, since we no longer need it held out) — and evaluate it on `X_test` for the first and only time.

In [ ]:
# Train the final model with the best hyperparameters, on the full training set
best_params = study.best_params
final_rf_classifier = RandomForestClassifier(random_state=42, n_jobs=-1, **best_params)
final_rf_classifier.fit(X_train, y_train)

# Evaluate the final model on the test set (used here for the first time)
final_test_predictions = final_rf_classifier.predict(X_test)

final_accuracy = accuracy_score(y_test, final_test_predictions)
print(f"\nAccuracy of the final model with best hyperparameters: {final_accuracy:.4f}")

print("\nClassification Report of the final model:")
class_names = full_ds.class_names
print(classification_report(y_test, final_test_predictions, target_names=class_names))

## Practice Problems

1. Compare the tuned Random Forest's test accuracy to the untuned baseline from earlier. How much did tuning actually help?
2. Swap `RandomForestClassifier` for a different classifier (e.g. `sklearn.svm.SVC` or `sklearn.linear_model.LogisticRegression`) on the same ResNet50 features, and write a new Optuna `objective` function that tunes its hyperparameters instead. Which classifier does better on these features?
3. Try reducing `n_trials` in `study.optimize()` from 150 down to 20. How much worse (if at all) is the best validation accuracy Optuna finds, and how much faster does it run? What does this tell you about the trade-off between search budget and result quality?
4. Instead of a single train/validation split inside `objective`, use `sklearn.model_selection.cross_val_score` with `cv=5` to score each trial. Does this change which hyperparameters Optuna considers best, or the final test accuracy?
5. This notebook extracts *frozen* ResNet50 features (no fine-tuning of ResNet50 itself). Compare this notebook's final accuracy to the CNN notebook's transfer-learning section (where a small trainable head was added on top of frozen VGG16 and trained end-to-end with backpropagation). Which approach — frozen feature extraction + Random Forest and Optuna, vs. frozen base + trained head — worked better here, and why might that be?